# Every optimized kernel on every image

Runs each optimized ZCU104 kernel on each image, one start for a whole capture, and writes
one report: each kernel's mean time and accuracy over the images, and every run. The
classes are checked against the images' labels, as in `classify.ipynb`.

This notebook sits at the top of the Jupyter folder. Next to it:

- each kernel's folder, `KERNEL_pynq/` (upload `KERNEL_pynq.zip` and run `!unzip -o KERNEL_pynq.zip`
  in a cell). The bitstream is taken from there: `justounetsimple_opt` from
  `justounetsimple_opt_pynq/justounetsimple_opt.bit`. A kernel without its folder is skipped.
- each image, `IMAGE.zip` or `IMAGE/` (cube.npy, labels.npy, meta.json), made on the PC or the server:

      python3 tools/image.py prepare CAPTURE-l1a.nc --labels CAPTURE-l1a_labels.npy --out trondheim
      source fpga.sh; fpga_zip trondheim

The report goes to `benchmark_results/benchmark_DATE.md`, every run also in a `.csv` beside it:
download them from Jupyter's file list. Shut down the other notebooks first: each image's cube
(157 MB for 598 x 1092) must fit in PYNQ's CMA. Run the cells from the top.

## 1. The kernels and the images

In [ ]:
import csv, json, re, time, zipfile
from datetime import datetime
from pathlib import Path
import numpy as np
from pynq import Overlay, allocate, Clocks

# The optimized ZCU104 kernels, each from its own KERNEL_pynq/ folder next to this notebook.
KERNELS = ['justoliunet_opt', 'justoliunet_bn_opt', 'justounetsimple_opt', 'justounetsimple_opt_tiled',
           'sp_unet_small_opt',
           'sp_unet_medium_opt', 'sp_unet_large_opt', 'sp_unet_small_opt_u8', 'sp_unet_medium_opt_u8',
           'sp_unet_large_opt_u8', 'spectralpointwise1d_opt', 'waveletcnn_opt']
# The images, each IMAGE.zip or IMAGE/ next to this notebook.
IMAGES = ['trondheim', 'hawkebay', 'lakealexandria', 'saldanha', 'maputo']
RESULTS = Path('benchmark_results')
BANDS, LANES = 120, 4       # in: H x W x 120 uint16; out: H x W x 4 int32, the 3 scores then the class
CLASSES = ('cloud', 'land', 'sea')

found = [k for k in KERNELS if (Path(f'{k}_pynq') / f'{k}.bit').exists()]
skipped = [k for k in KERNELS if k not in found]
if not found:
    raise FileNotFoundError(f'no KERNEL_pynq/KERNEL.bit for any of {KERNELS} in {Path(".").resolve()}')

def image_folder(name):
    # IMAGE/, unpacked from IMAGE.zip when the zip is newer than what is unpacked.
    folder, archive = Path(name), Path(f'{name}.zip')
    if archive.exists() and (not (folder / 'cube.npy').exists()
                             or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
        with zipfile.ZipFile(archive) as z:
            z.extractall('.')
    for file in ('cube.npy', 'labels.npy'):
        if not (folder / file).exists():
            raise FileNotFoundError(f'no {folder / file}: make the image with tools/image.py prepare --labels')
    return folder

# Every image before the first run, so a missing one does not stop the run halfway.
folders = {image: image_folder(image) for image in IMAGES}
print('kernels:', ', '.join(found))
if skipped:
    print('skipped (no KERNEL_pynq/KERNEL.bit):', ', '.join(skipped))
print('images: ', ', '.join(IMAGES))

## 2. One kernel

In [ ]:
class Kernel:
    """An optimized kernel: its bitstream on the FPGA, its registers from its x..._hw.h."""

    def __init__(self, name):
        folder = Path(f'{name}_pynq')
        info = json.loads((folder / 'kernel.json').read_text())
        self.name, self.model, self.patch = name, info['model'], info['patch']   # patch: None for a 1D network
        header = next(folder.glob('x*_hw.h')).read_text()
        def register(reg):
            return int(re.search(rf'_ADDR_{reg}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
        self.ap_ctrl, self.height, self.width = register('AP_CTRL'), register('HEIGHT_DATA'), register('WIDTH_DATA')
        self.dins = [int(a, 16) for a in re.findall(r'_ADDR_DIN\d+_DATA\s+(0x[0-9a-fA-F]+)', header)]
        self.dout = register('DOUT_DATA')
        # The kernel's own count of its clock cycles, from its start until the last score is in memory.
        self.cycles = register('CYCLES_DATA') if '_ADDR_CYCLES_DATA' in header else None
        self.ip = Overlay(str(folder / f'{name}.bit')).kernel   # also sets the FPGA clock from the .hwh
        self.mhz = Clocks.fclk0_mhz

    def run(self, cube, scores, timeout=60):
        # The whole cube in one start: (seconds from the start bit to the done bit as the ARM
        # saw them, seconds by the kernel's own count or None).
        h, w, _ = cube.shape
        for offset, address in [(d, cube.physical_address) for d in self.dins] + [(self.dout, scores.physical_address)]:
            self.ip.write(offset, address & 0xFFFFFFFF)
            self.ip.write(offset + 4, address >> 32)
        self.ip.write(self.height, h)
        self.ip.write(self.width, w)
        t = time.perf_counter()
        self.ip.write(self.ap_ctrl, 1)
        deadline = time.monotonic() + timeout
        while not self.ip.read(self.ap_ctrl) & 0x2:     # done
            if time.monotonic() > deadline:
                raise TimeoutError(f'{self.name} did not finish in {timeout} s')
        fpga = time.perf_counter() - t
        own = None if self.cycles is None else self.ip.read(self.cycles) / (self.mhz * 1e6)
        return fpga, own

    def units(self, h, w):
        # What the cycles are given per: 32 x 32 patches for a 2D network, pixels for a 1D one.
        return (-(-h // self.patch) * -(-w // self.patch), 'patch') if self.patch else (h * w, 'pixel')

## 3. Every kernel on every image

Image by image: the cube goes from the SD card into DDR once, then each kernel's bitstream
is loaded and runs on it. **Wall clock**: from the start until the ARM can read the classes.
**FPGA**: from the start bit to the done bit, as the ARM sees it. **Own count**: the kernel's
cycle counter, the kernel alone. A run that fails is written down and the others go on.

In [ ]:
rows = []
for image, folder in folders.items():
    t0 = time.monotonic()
    cube = np.load(folder / 'cube.npy', mmap_mode='r')
    if cube.dtype != np.uint16:                       # an image folder from before: counts as floats
        cube = np.rint(cube).astype(np.uint16)
    labels = np.load(folder / 'labels.npy')
    if labels.shape != cube.shape[:2]:
        raise ValueError(f'{image}: labels.npy is {labels.shape}, the cube {cube.shape[:2]}')
    H, W, _ = cube.shape
    known = labels < 3                                # 3 and above: unlabeled
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
    cube_in[:] = cube
    cube_in.flush()                                   # into DDR, not just the CPU cache
    scores_out = allocate(shape=(H, W, LANES), dtype=np.int32)
    print(f'{image}: {H} x {W} pixels, {known.sum()} labelled, loaded in {time.monotonic() - t0:.1f} s')
    for name in found:
        row = {'image': image, 'kernel': name, 'height': H, 'width': W, 'labelled': int(known.sum()), 'error': None}
        try:
            kernel = Kernel(name)
            count, unit = kernel.units(H, W)
            scores_out.fill(-1)                       # a class the kernel did not write counts as wrong
            scores_out.flush()
            t = time.perf_counter()
            fpga, own = kernel.run(cube_in, scores_out)
            scores_out.invalidate()                   # read what the kernel wrote
            classes = np.asarray(scores_out)[..., 3]
            wall = time.perf_counter() - t
            hit = classes == labels
            row.update(model=kernel.model, mhz=kernel.mhz, units=count, unit=unit,
                       wall_ms=1000 * wall, fpga_ms=1000 * fpga, own_ms=None if own is None else 1000 * own,
                       cycles_per_unit=None if own is None else own * kernel.mhz * 1e6 / count,
                       correct=int(hit[known].sum()), accuracy=float(hit[known].mean()),
                       intersection=[int(((classes == c) & (labels == c))[known].sum()) for c in range(3)],
                       union=[int(((classes == c) | (labels == c))[known].sum()) for c in range(3)])
            print(f'  {name:24s} {row["fpga_ms"]:9.2f} ms FPGA'
                  + ('' if own is None else
                     f', {row["own_ms"]:9.2f} ms by its count ({row["cycles_per_unit"]:.2f} cycles a {unit})')
                  + f', {row["accuracy"]:.2%}')
        except Exception as error:
            row['error'] = f'{type(error).__name__}: {error}'
            print(f'  {name:24s} FAILED: {row["error"]}')
        rows.append(row)
    cube_in.freebuffer()
    scores_out.freebuffer()
    del cube

## 4. The report

Each kernel's mean over the images it ran on: the times, the cycles a patch or pixel, the
accuracy (the mean of the images', and over all their labelled pixels at once) and each
class's IoU over all of them. Written to `benchmark_results/` and printed here.

In [ ]:
def mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

def show(value, spec):
    return '-' if value is None or value != value else format(value, spec)

summary = []
for name in found:
    runs = [r for r in rows if r['kernel'] == name and r['error'] is None]
    if not runs:
        continue
    inter = np.sum([r['intersection'] for r in runs], axis=0)
    union = np.sum([r['union'] for r in runs], axis=0)
    iou = [i / u if u else float('nan') for i, u in zip(inter, union)]
    summary.append({'kernel': name, 'mhz': runs[0]['mhz'], 'images': len(runs), 'unit': runs[0]['unit'],
                    'cycles_per_unit': mean(r['cycles_per_unit'] for r in runs),
                    'own_ms': mean(r['own_ms'] for r in runs), 'fpga_ms': mean(r['fpga_ms'] for r in runs),
                    'wall_ms': mean(r['wall_ms'] for r in runs),
                    'ns_per_pixel': mean(None if r['own_ms'] is None else 1e6 * r['own_ms'] / (r['height'] * r['width'])
                                         for r in runs),
                    'accuracy': mean(r['accuracy'] for r in runs),
                    'pooled': sum(r['correct'] for r in runs) / sum(r['labelled'] for r in runs),
                    'iou': iou, 'miou': float(np.nanmean(iou))})

stamp = datetime.now().strftime('%Y-%m-%d_%H%M')
shapes = {r['image']: (r['height'], r['width']) for r in rows}
lines = [f'# Optimized kernels on the ZCU104, {stamp}', '',
         'Images: ' + ', '.join(f'{i} ({h} x {w})' for i, (h, w) in shapes.items()) + '.', '',
         f'## Mean over the {len(IMAGES)} images', '',
         '| Kernel | MHz | Images | Cycles (own count) | Own count (ms) | FPGA (ms) | Wall clock (ms) '
         '| ns a pixel | Accuracy (mean) | Accuracy (all pixels) | IoU cloud | IoU land | IoU sea | mIoU |',
         '|' + ' --- |' * 14]
for s in summary:
    lines.append(f'| `{s["kernel"]}` | {s["mhz"]:.1f} | {s["images"]} '
                 f'| {show(s["cycles_per_unit"], ".2f")} a {s["unit"]} | {show(s["own_ms"], ".3f")} '
                 f'| {s["fpga_ms"]:.3f} | {s["wall_ms"]:.3f} | {show(s["ns_per_pixel"], ".2f")} '
                 f'| {s["accuracy"]:.2%} | {s["pooled"]:.2%} | '
                 + ' | '.join(show(v, '.3f') for v in s['iou']) + f' | {show(s["miou"], ".3f")} |')
lines += ['', '## Every run', '',
          '| Image | Kernel | Own count (ms) | FPGA (ms) | Wall clock (ms) | Cycles | Accuracy |',
          '|' + ' --- |' * 7]
for r in rows:
    if r['error'] is None:
        lines.append(f'| {r["image"]} | `{r["kernel"]}` | {show(r["own_ms"], ".3f")} | {r["fpga_ms"]:.3f} '
                     f'| {r["wall_ms"]:.3f} | {show(r["cycles_per_unit"], ".2f")} a {r["unit"]} | {r["accuracy"]:.2%} |')
    else:
        lines.append(f'| {r["image"]} | `{r["kernel"]}` | failed: {r["error"]} | | | | |')
if skipped:
    lines += ['', 'Skipped, no `KERNEL_pynq/KERNEL.bit`: ' + ', '.join(f'`{k}`' for k in skipped) + '.']
report = '\n'.join(lines) + '\n'

RESULTS.mkdir(exist_ok=True)
(RESULTS / f'benchmark_{stamp}.md').write_text(report)
fields = ['image', 'kernel', 'model', 'mhz', 'height', 'width', 'units', 'unit', 'own_ms', 'fpga_ms', 'wall_ms',
          'cycles_per_unit', 'labelled', 'correct', 'accuracy', 'iou_cloud', 'iou_land', 'iou_sea', 'error']
with open(RESULTS / f'benchmark_{stamp}.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fields, extrasaction='ignore')
    writer.writeheader()
    for r in rows:
        ious = {} if r['error'] else {f'iou_{c}': (i / u if u else None)
                                      for c, i, u in zip(CLASSES, r['intersection'], r['union'])}
        writer.writerow({**r, **ious})
print(report)
print(f'written: {RESULTS / f"benchmark_{stamp}.md"} and its .csv')